# Mario RL Agent Experiments\n\nThis notebook is for testing the environment, visualizing frames, and debugging the RL agent implementation.\n\n## Sections:\n1. Environment Testing\n2. Frame Visualization\n3. Agent Debugging\n4. Training Monitoring

In [ ]:
# Import necessary libraries\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport sys\nimport os\n\n# Add project root to path\nsys.path.append(os.path.join(os.path.dirname(__file__), '..'))\n\n# Import our custom modules\nfrom envs.mario_env import make_mario_env\nfrom agents.dqn_agent import DQNAgent\nfrom models.cnn import MarioCNN, DQNNet, PPONet\n\n# For Jupyter display\n%matplotlib inline

In [ ]:
# Test the Mario environment\nprint("Creating Mario environment...")\nenv = make_mario_env(\n    env_name='SuperMarioBros-1-1-v0',\n    action_type='simple',\n    frame_stack=4,\n    frame_size=(84, 84),\n    frame_skip=4\n)\n\nprint(f"Observation space: {env.observation_space}")\nprint(f"Action space: {env.action_space}")\n\n# Reset and get initial state\nstate, info = env.reset()\nprint(f"Initial state shape: {state.shape}")\nprint(f"Initial state dtype: {state.dtype}")\nprint(f"Initial state min/max: {state.min()}/{state.max()}")\n\n# Take a few random steps and visualize\nfig, axes = plt.subplots(2, 2, figsize=(8, 8))\naxes = axes.flatten()\n\nfor i in range(4):\n    action = env.action_space.sample()\n    state, reward, done, truncated, info = env.step(action)\n    \n    # Display the first frame in the stack (most recent)\n    frame_to_show = state[0]  # First channel = most recent frame\n    axes[i].imshow(frame_to_show, cmap='gray')\n    axes[i].set_title(f'Step {i+1}: Reward={reward:.2f}')\n    axes[i].axis('off')\n    \n    if done or truncated:\n        break\n\nplt.tight_layout()\nplt.show()\n\nprint(f"Total reward after {i+1} steps: {reward:.2f}")\nenv.close()

In [ ]:
# Test the CNN architecture\nimport torch\nimport torch.nn.functional as F\n\nprint("Testing CNN architecture...")\ncnn = MarioCNN(input_shape=(4, 84, 84))\nprint(f"CNN model:\n{cnn}")\n\n# Create dummy input\ndummy_input = torch.randn(1, 4, 84, 84, dtype=torch.uint8)\nprint(f"Input shape: {dummy_input.shape}")\n\n# Forward pass\nwith torch.no_grad():\n    features = cnn(dummy_input)\n    print(f"Output features shape: {features.shape}")\n    print(f"Output features mean/std: {features.mean():.4f}/{features.std():.4f}")\n\n# Test DQN network\nprint("\nTesting DQN network...")\ndqn_net = DQNNet(input_shape=(4, 84, 84), n_actions=7)\nwith torch.no_grad():\n    q_values = dqn_net(dummy_input)\n    print(f"Q-values shape: {q_values.shape}")\n    print(f"Q-values sample: {q_values[0][:5]}")\n\n# Test PPO network\nprint("\nTesting PPO network...")\nppo_net = PPONet(input_shape=(4, 84, 84), n_actions=7)\nwith torch.no_grad():\n    policy_logits, value = ppo_net(dummy_input)\n    print(f"Policy logits shape: {policy_logits.shape}")\n    print(f"Value shape: {value.shape}")\n    print(f"Policy logits sample: {policy_logits[0][:5]}")\n    print(f"Value sample: {value[0].item():.4f}")

In [ ]:
# Test the DQN agent\nprint("Testing DQN agent...")\nagent = DQNAgent(input_shape=(4, 84, 84), n_actions=7)\nprint(f"Agent created with epsilon: {agent.epsilon}")\n\n# Test action selection\ndummy_state = np.random.randint(0, 255, size=(4, 84, 84), dtype=np.uint8)\naction = agent.select_action(dummy_state)\nprint(f"Selected action: {action}")\n\n# Test experience storage\nagent.store_experience(dummy_state, action, 1.0, dummy_state, False)\nprint(f"Buffer size after storing: {len(agent.replay_buffer)}")\n\n# Test learning (placeholder)\nloss = agent.learn()\nprint(f"Loss from learning step: {loss}")\n\n# Test epsilon update\nagent.update_epsilon()\nprint(f"Epsilon after update: {agent.epsilon}")